In [6]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("API_KEY")

In [ ]:
url = "https://api.semanticscholar.org/graph/v1/paper/search"

params = {
    "query": "machine learning",
    "limit": 5,
    "fields": "title,authors,year,citationCount,abstract,paperId"
}

headers = {
    "User-Agent": "Alyssa Wilson (flutelys@byu.edu)",
    "x-api-key": api_key
}

response = requests.get(url, params=params, headers=headers)
data = response.json()

print(data.keys())
print(data)

dict_keys(['total', 'offset', 'next', 'data'])
{'total': 7354649, 'offset': 0, 'next': 5, 'data': [{'paperId': '53c9f3c34d8481adaf24df3b25581ccf1bc53f5c', 'title': 'Physics-informed machine learning', 'year': 2021, 'citationCount': 6020, 'openAccessPdf': {'url': 'https://www.osti.gov/biblio/2282016', 'status': 'GREEN', 'license': 'other-oa', 'disclaimer': "Notice: The following paper fields have been elided by the publisher: {'abstract'}. Paper or abstract available at https://api.unpaywall.org/v2/10.1038/s42254-021-00314-5?email=<INSERT_YOUR_EMAIL> or https://doi.org/10.1038/s42254-021-00314-5, which is subject to the license by the author or copyright owner provided with this content. Please go to the source to verify the license and copyright information for your use."}, 'authors': [{'authorId': '1720124', 'name': 'G. Karniadakis'}, {'authorId': '3439407', 'name': 'I. Kevrekidis'}, {'authorId': '2149373363', 'name': 'Lu Lu'}, {'authorId': '3410970', 'name': 'P. Perdikaris'}, {'autho

In [8]:
papers = data["data"]

for paper in papers:
    title = paper["title"]
    authors = [a["name"] for a in paper.get("authors", [])]
    year = paper.get("year")
    citations = paper.get("citationCount")

In [ ]:
rows = []

for paper in papers:
    time.sleep(1)  # To respect API rate limits
    paper_id = paper.get("paperId")

    paper_url = f"https://api.semanticscholar.org/graph/v1/paper/{paper_id}"
    paper_params = {"fields": "abstract,influentialCitationCount"}

    paper_resp = requests.get(paper_url, headers=headers, params=paper_params)
    paper_data = paper_resp.json()

    abstract = paper_data.get("abstract")
    influential = paper_data.get("influentialCitationCount")

    rows.append({
        "title": paper.get("title"),
        "authors": ", ".join(a["name"] for a in paper.get("authors", [])),
        "year": paper.get("year"),
        "citations": paper.get("citationCount"),
        "abstract": abstract or paper.get("abstract"),
        "paperId": paper.get("paperId"),
        "isInfluential": influential
    })
    

df = pd.DataFrame(rows)
df

,title,authors,year,citations,abstract,paperId,isInfluential
0,Physics-informed machine learning,"G. Karniadakis, I. Kevrekidis, Lu Lu, P. Perdi...",2021,6020,NaN,53c9f3c34d8481adaf24df3b25581ccf1bc53f5c,165
1,"Machine Learning: Algorithms, Real-World Appli...",Iqbal H. Sarker,2021,4176,In the current age of the Fourth Industrial Re...,7872f34e2a164c5cf3c34a7a7433dc3342b6c7ea,154
2,Fashion-MNIST: a Novel Image Dataset for Bench...,"Han Xiao, Kashif Rasul, Roland Vollgraf",2017,10389,"We present Fashion-MNIST, a new dataset compri...",f9c602cc436a9ea2f9e7db48c77d924e09ce3c32,1639
3,A Survey on Bias and Fairness in Machine Learning,"Ninareh Mehrabi, Fred Morstatter, N. Saxena, K...",2019,5619,With the widespread use of artificial intellig...,0090023afc66cd2741568599057f4e82b566137c,370
4,Stop explaining black box machine learning mod...,C. Rudin,2018,8315,NaN,bc00ff34ec7772080c7039b17f7069a2f7df0889,427


In [47]:
all_rows = []

for offset in range(0, 500, 10):
    time.sleep(1)  # respect rate limits for search requests
    
    params["offset"] = offset
    params["limit"] = 10

    response = requests.get(url, params=params, headers=headers)
    data = response.json()

    papers = data.get("data", [])

    for paper in papers:
        time.sleep(1)  # respect rate limits for per-paper requests
        
        paper_id = paper.get("paperId")

        # --- Enrichment call ---
        paper_url = f"https://api.semanticscholar.org/graph/v1/paper/{paper_id}"
        paper_params = {"fields": "abstract,influentialCitationCount"}

        paper_resp = requests.get(paper_url, headers=headers, params=paper_params)
        paper_data = paper_resp.json()

        # --- Extract enriched fields ---
        abstract = paper_data.get("abstract")
        influential = paper_data.get("influentialCitationCount")

        # --- Append row ---
        all_rows.append({
            "title": paper.get("title"),
            "authors": ", ".join(a.get("name", "") for a in paper.get("authors", [])),
            "year": paper.get("year"),
            "citations": paper.get("citationCount"),
            "abstract": abstract or paper.get("abstract"),
            "paperId": paper_id,
            "influential_citations": influential
        })

In [48]:
df = pd.DataFrame(all_rows)

In [49]:
df

,title,authors,year,citations,abstract,paperId,influential_citations
0,Physics-informed machine learning,"G. Karniadakis, I. Kevrekidis, Lu Lu, P. Perdi...",2021,6020,NaN,53c9f3c34d8481adaf24df3b25581ccf1bc53f5c,165.0
1,"Machine Learning: Algorithms, Real-World Appli...",Iqbal H. Sarker,2021,4176,In the current age of the Fourth Industrial Re...,7872f34e2a164c5cf3c34a7a7433dc3342b6c7ea,154.0
2,Fashion-MNIST: a Novel Image Dataset for Bench...,"Han Xiao, Kashif Rasul, Roland Vollgraf",2017,10389,"We present Fashion-MNIST, a new dataset compri...",f9c602cc436a9ea2f9e7db48c77d924e09ce3c32,1639.0
3,A Survey on Bias and Fairness in Machine Learning,"Ninareh Mehrabi, Fred Morstatter, N. Saxena, K...",2019,5619,With the widespread use of artificial intellig...,0090023afc66cd2741568599057f4e82b566137c,370.0
4,Stop explaining black box machine learning mod...,C. Rudin,2018,8315,NaN,bc00ff34ec7772080c7039b17f7069a2f7df0889,427.0
...,...,...,...,...,...,...,...
495,A survey of techniques for internet traffic cl...,"Thuy T. T. Nguyen, G. Armitage",2008,1719,NaN,7da323e7103245eeaed32367c46abe3f4913df86,87.0
496,Machine Learning and the Profession of Medicine.,"Alison M Darcy, A. Louie, L. Roberts",2016,379,NaN,9f86366feecbcfdf6c5be165fcf38c679164cc89,6.0
497,Python Machine Learning,,2015,667,NaN,9ec1cb983d6d475bf0fc2879ea1a7e31201d8c37,61.0
498,A review of automatic selection methods for ma...,G. Luo,2016,333,NaN,0bf08d3dcc1f4a3b7b4f9a68f9c980c0a3f4ed2a,11.0
